# Study 936 — Tolerance Bands ⚖️

**Does rebalancing on 5/25 tolerance bands beat rebalancing on the calendar?**

Every allocation guide eventually reaches the rebalancing chapter, and the answer it gives
is almost always the same: don't trade on the calendar, trade on **tolerance bands** — act
when a sleeve is off target by *5 percentage points absolute or 25% of its own weight*,
whichever binds first. The promise: better risk-adjusted returns **and** less turnover,
because you only trade when the book has actually moved.

We test it on **SPY / IEF / GLD** with **BIL** as the cash leg,
2007-05-30 → 2026-06-30 (4,802 days of total-return closes), one execution lag
(breach seen at the close of day *t*, traded at the close of *t+1*), one-way cost charged
on traded notional, every Sharpe **excess-of-cash**.

*Real-tape numbers below are the frozen headline from `docs/results.md`
(returns fingerprint `958f652ce4c9`, as-of 2026-06-30). The live cells at the end run the
**synthetic** control and are labelled as such.*


## 1. The question, stated fairly

A 60/40 book does not stay 60/40. Stocks out-run bonds, the equity sleeve grows, and within a few years you own a portfolio you never chose. So you rebalance. The only real decision is **when**: on a date (every January; every quarter), or on a **condition** (whenever the equity sleeve is more than 5 points off).

The condition sounds obviously better. It carries information about your actual book; a date carries none. Let's see how much that is worth.

## 2. The answer: about a third of one percent of a Sharpe point

Here are the four schedules on the same 60/40 SPY/IEF book, all excess-of-cash, all costed.

In [1]:
# Frozen real-tape headline (docs/results.md). Nothing is recomputed here.
R = {'a_drift_s': 0.6788, 'a_ann_s': 0.6673, 'a_qtr_s': 0.657, 'a_bnd_s': 0.6412, 'a_drift_c': 7.23, 'a_ann_c': 6.79, 'a_qtr_c': 6.84, 'a_bnd_c': 6.88, 'a_ann_to': 0.079, 'a_qtr_to': 0.157, 'a_bnd_to': 0.114, 'a_diff_ba': -0.0261, 'a_t_ba': 0.82, 'a_ci_lo': -0.0674, 'a_ci_hi': 0.0142, 'drag50_ann': 3.94, 'drag50_qtr': 7.86, 'drag50_bnd': 5.69}
rows = [('never rebalance', R['a_drift_s'], R['a_drift_c'], 0.0),
        ('annual',          R['a_ann_s'],   R['a_ann_c'],   R['a_ann_to']),
        ('quarterly',       R['a_qtr_s'],   R['a_qtr_c'],   R['a_qtr_to']),
        ('5/25 bands',      R['a_bnd_s'],   R['a_bnd_c'],   R['a_bnd_to'])]
print(f"{'schedule':<16}{'excess Sharpe':>15}{'CAGR':>9}{'traded/yr':>12}")
for name, s, c, to in rows:
    print(f'{name:<16}{s:>15.3f}{c:>8.2f}%{to:>11.1%}')
print()
print(f"bands minus annual: {R['a_diff_ba']:+.3f} Sharpe  "
      f"(HAC t = {R['a_t_ba']:+.2f}, 95% CI "
      f"[{R['a_ci_lo']:+.3f}, {R['a_ci_hi']:+.3f}])")

schedule          excess Sharpe     CAGR   traded/yr
never rebalance           0.679    7.23%       0.0%
annual                    0.667    6.79%       7.9%
quarterly                 0.657    6.84%      15.7%
5/25 bands                0.641    6.88%      11.4%

bands minus annual: -0.026 Sharpe  (HAC t = +0.82, 95% CI [-0.067, +0.014])


All four are within **0.038 of a Sharpe point** of each other over nineteen years. The band rule comes out **-0.026** against annual — the *wrong* side of zero for the claim, and nowhere near big enough to call. The confidence interval **[-0.067, +0.014]** is not wide because the test is weak; it is narrow, and it is centred on nothing.

> 🔬 **For the quants:** the sign is split. The band book earns a *fractionally higher* mean daily return (which is why the HAC *t* on the return difference is **+0.82**, positive) and runs *more* volatility (11.40% vs 10.71%), so its Sharpe lands fractionally lower. Both effects are noise; they happen to point opposite ways.

## 3. The published claim, and what actually turned up

The strongest version of the band case in print — Daryanani's 2008 *Opportunistic Rebalancing* — reports roughly **0.5 percentage points a year** for bands over annual calendar rebalancing. On this tape the gap is **+0.09 pp/yr**: about a fifth of the claim, and well inside the noise. The 3-asset 50/30/20 SPY/IEF/GLD book gives **-0.028** Sharpe (*t* = +0.80); the 23-year SPY/IEF cross-check gives **-0.032** (*t* = +1.04) — that one is measured *gross of cash*, because no tradable cash ETF existed before 2007, so its like-for-like comparator is the **-0.034** you get by running the headline window the same way. Three shots, same nothing.

## 4. Costs are not the argument

The usual defence of bands is that they save you turnover. They do — and it does not matter, because a 60/40 ETF book barely trades under *any* schedule.

In [2]:
sched = [('annual', R['a_ann_to'], R['drag50_ann']),
         ('quarterly', R['a_qtr_to'], R['drag50_qtr']),
         ('5/25 bands', R['a_bnd_to'], R['drag50_bnd'])]
print('traded notional per year, and the annual drag at a PUNITIVE 50 bps one-way')
print('(ten times a realistic SPY/IEF spread)')
print()
for name, turn, drag in sched:
    print(f'{name:<12} {turn:>6.1%} of NAV traded per year  ->  {drag:>4.1f} bp/yr of drag')
gap = R['drag50_qtr'] - R['drag50_ann']
print(f'\nThe gap between the most and least expensive schedule: {gap:.1f} bp/yr.')

traded notional per year, and the annual drag at a PUNITIVE 50 bps one-way
(ten times a realistic SPY/IEF spread)

annual         7.9% of NAV traded per year  ->   3.9 bp/yr of drag
quarterly     15.7% of NAV traded per year  ->   7.9 bp/yr of drag
5/25 bands    11.4% of NAV traded per year  ->   5.7 bp/yr of drag

The gap between the most and least expensive schedule: 3.9 bp/yr.


Four basis points a year, at ten times realistic cost. The turnover argument is true and irrelevant.

> 🔬 **For the quants:** this is why the cost sweep in `docs/results.md` is flat from 0 to 50 bps — bands minus annual moves from -0.0260 to -0.0273. Friction is simply not the binding constraint at ETF spreads; it would be at 1990s mutual-fund loads, which is the era the folklore was formed in.

## 5. What rebalancing *does* buy — and here bands genuinely win

There is one axis on which the schedules differ enormously, and it has nothing to do with return. Left alone, the 60/40 book ended the sample holding **84.6% equity** — a portfolio nobody chose, with a completely different risk profile from the one that was signed off.

| schedule | equity sleeve stayed between | traded/yr |
|---|---|---|
| never rebalance | 35.7% – 85.0% | 0.0% |
| annual | 41.8% – 67.2% | 7.9% |
| quarterly | 48.1% – 66.6% | 15.7% |
| **5/25 bands** | **53.6% – 65.2%** | **11.4%** |

The band rule holds the **tightest envelope** for **27% less traded notional than quarterly**. Two things that claim does *not* say, and that the table above will tell you if you read it carefully: annual rebalancing is **cheaper still** (7.9% a year — it just lets the book roam nearly twice as far), and quarterly sits **closer to target on an average day**. What bands minimise is the *worst* excursion, which is the number a risk mandate is written on.

That is a real, reproducible advantage — it is just an advantage in *governance*, not in return. If your reason for rebalancing is 'I want to still own the portfolio I chose', bands are the best tool here. If your reason is 'it will make me more money', the tape says no.

## 6. Live check — the machinery is not broken (offline synthetic)

**This cell runs on a synthetic tape, not on the real one.** We build a world where the two sleeves' relative performance genuinely mean-reverts (a 60-day half-life) — exactly the world in which trading on dispersion *should* pay — and a null world where dispersion is a random walk. The band rule must win in the first and not in the second.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from rebal_bands import data, strategy as st
for tag, ss in [('planted (mean-reverting)', 1.0), ('null (random walk)', 0.0)]:
    d = st.synthetic_detect(data.synthetic_panel(signal_strength=ss, seed=936)[0])
    print(f"{tag:26s}: bands - annual Sharpe {d['diff_bands_annual']:+.4f}  "
          f"(HAC t {d['t_bands_annual']:+.2f})")

planted (mean-reverting)  : bands - annual Sharpe +0.0414  (HAC t +3.01)


null (random walk)        : bands - annual Sharpe +0.0013  (HAC t -0.05)


The detector fires on the planted world (**+0.0414**, HAC *t* >= 2 on **7/8** seeds) and stays quiet on the null (**+0.0044**, **0/8**). So the flat real-tape answer is a fact about stocks, bonds and gold — their relative performance does not mean-revert on the horizon a 5-point band operates over — not a bug in the backtest.

## Verdict

- **Signal — None.** Bands minus annual is **-0.026** excess Sharpe (HAC *t* = +0.82), CI **[-0.067, +0.014]**, and the same non-result holds on the 3-asset book, in both eras, at every cost from 0 to 50 bps, and across band widths from 2/10 to 10/50. The famous ~0.5 pp/yr advantage comes out at **+0.09 pp/yr**.
- **Tradability — Mirage.** Nothing to bank. The schedules differ by under 4 bp/yr of cost even at ten times realistic spreads — and tax, the one cost we do *not* model, runs against the busier schedules, not for them.
- **What is real.** Rebalance on *something*: the do-nothing book finished at 85% equity. Among schedules that do the job, 5/25 bands give the tightest *worst-case* weight envelope for less trading than quarterly — not the least trading outright (annual is cheaper), and not the closest average tracking (quarterly is closer). Pick your rebalancing rule for the portfolio you want to still own, not for the return you hope to add.